# 1. Penentuan Topik & Knowledge Base
1. Pilih salah satu topik knowledge base yang akan digunakan sebagai dasar pembuatan chatbot RAG.
2. Buat text cell terpisah, lalu jelaskan secara singkat ruang lingkup informasi yang dibahas dalam knowledge base tersebut.


- a. topic yang akan saya angkat mengenai faktor social ekonomi dan kualitas lingkungan terhadap kemiskinan di indonesia
- b. saya akan menggunakan llm dengan menggunakan data yang di ambil dari google scolar menggunakan 1 pdf sebagai referensi yang aku ekstrak ke dalam bentuk teks yang di mana isi nya mengenai apakah faktor ekonomi dan faktor social mempengaruhi kemiskinan di indonesia 


# 2. Penyusunan Knowledge Base
1. Siapkan knowledge base dalam bentuk teks dari sumber referensi yang relevan.
2. Pastikan knowledge base yang dibuat dapat diperbarui, baik dengan menambahkan maupun mengganti konten.


In [2]:
knowledge_base = [
    """
    Kemiskinan di Indonesia merupakan permasalahan klasik yang kompleks dan masih menjadi tantangan serius.
    Upaya penanggulangan kemiskinan membutuhkan kebijakan yang berkelanjutan karena kemiskinan tidak dapat
    diselesaikan dalam waktu singkat. Indonesia juga berkomitmen mengurangi kemiskinan melalui program
    pembangunan nasional dan Sustainable Development Goals (SDGs), dengan target penurunan tingkat kemiskinan.
    Namun, target tersebut belum sepenuhnya tercapai karena adanya berbagai faktor yang mempengaruhi.
    """,

    """
    Faktor ekonomi seperti Produk Domestik Regional Bruto (PDRB) memiliki hubungan dengan kemiskinan.
    Secara umum, peningkatan PDRB dapat menurunkan tingkat kemiskinan karena mencerminkan pertumbuhan ekonomi.
    Namun, dalam beberapa kasus, pengaruhnya tidak signifikan. Hal ini menunjukkan bahwa pertumbuhan ekonomi
    saja tidak cukup, melainkan harus merata agar dapat dirasakan oleh seluruh masyarakat, terutama kelompok miskin.
    """,

    """
    Faktor sosial seperti kualitas pembangunan manusia yang diukur melalui Indeks Pembangunan Manusia (IPM)
    memiliki pengaruh signifikan dalam menurunkan kemiskinan. Semakin tinggi kualitas pendidikan, kesehatan,
    dan kesejahteraan masyarakat, maka tingkat kemiskinan akan semakin menurun. Hal ini menunjukkan bahwa
    peningkatan kualitas sumber daya manusia sangat penting dalam memutus lingkaran kemiskinan.
    """,

    """
    Faktor lingkungan juga berpengaruh terhadap kemiskinan. Kualitas lingkungan yang buruk dapat meningkatkan
    tingkat kemiskinan karena masyarakat miskin sering berada di wilayah dengan kondisi lingkungan yang tidak
    mendukung. Namun, dalam beberapa kondisi, peningkatan kualitas lingkungan juga dapat berdampak pada
    meningkatnya kemiskinan jika kebijakan lingkungan membatasi akses masyarakat terhadap sumber daya alam.
    """,

    """
    Kemiskinan juga dipengaruhi oleh lingkaran setan kemiskinan (vicious circle of poverty), yaitu kondisi di mana
    rendahnya produktivitas menyebabkan pendapatan rendah, sehingga tabungan dan investasi juga rendah.
    Hal ini mengakibatkan keterbelakangan ekonomi yang terus berulang. Faktor seperti rendahnya kualitas
    sumber daya manusia dan ketidaksempurnaan pasar turut memperkuat kondisi ini.
    """
]

def add_to_knowledge_base(new_text: str):
    knowledge_base.append(new_text)
    print (f'knowledge_base updated. total dokumen : {len(knowledge_base)}')
    
print(f'knowledge_base siap. total dokumen : {len(knowledge_base)}')

knowledge_base siap. total dokumen : 5


# 3. Desain Sistem RAG
1. Tentukan komponen utama sistem RAG yang akan digunakan, meliputi:    
    - Model bahasa (LLM)
    - Model embedding
    - Media penyimpanan embedding (vector database)
2. Buat text cell terpisah, dan jelaskan alasan pemilihan setiap komponen yang digunakan dalam sistem.


## komponen yang di gunakan
| Komponen         | Pilihan                          | Alasan                                                                 |
|------------------|----------------------------------|------------------------------------------------------------------------|
| **LLM**          | Groq (llama-3.1-8b-instant)            | Gratis, kecepatan respon tinggi                     |
| **Embedding**    | Ollama (nomic-embed-text)        | Berjalan lokal, tidak ada biaya, performa embedding solid              |
| **Vector Store** | FAISS(CPU)                   | Ringan, tidak perlu server eksternal, cocok untuk skala notebook       |


# 4. Implementasi Aplikasi RAG
1. Bangun aplikasi chatbot RAG yang mampu:
    - Menerima input pertanyaan dari pengguna
    - Mengambil informasi relevan dari knowledge base
    - Menghasilkan jawaban berdasarkan hasil retrieval
2. Pastikan sistem mendukung pembaruan knowledge base yang telah disiapkan.


## melakukan persiapan

In [ ]:
import os                                   
from dotenv import load_dotenv              
import langchain_core
from langchain_groq import ChatGroq                                     
from langchain_ollama import OllamaEmbeddings                           
from langchain_community.vectorstores import FAISS                      
from langchain_text_splitters import RecursiveCharacterTextSplitter      
from langchain_core.documents import Document                           
from langchain_core.prompts import ChatPromptTemplate                   
from langchain_core.output_parsers import StrOutputParser               
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
    
GROQ_API_KEY = os.getenv("GROQ_API_KEY") # di simpan di file .env

if not GROQ_API_KEY:
    print('api tidak di temukan')
else:
    print('api key berhasil di termukan')

d:\bahasa_pemograman\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


api key berhasil di termukan


## build vector_store

In [4]:
def build_vector_store(kb: list, chunk_size: int = 500, chunk_overlap: int = 50):
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    
    raw_docs = [Document(page_content=text) for text in kb]
    
    chunks = splitter.split_documents(raw_docs)
    print(f'jumlah chunk yang dihasilkan: {len(chunks)}')
    
    embedding = OllamaEmbeddings(model='nomic-embed-text')
    
    vector_store = FAISS.from_documents(chunks, embedding)

    print('vector berhasil di bangun')
    return vector_store

vector_store = build_vector_store(knowledge_base)

jumlah chunk yang dihasilkan: 6
vector berhasil di bangun


## build RAG Chain

In [9]:
def build_rag(vs: FAISS, model_name: str = 'llama-3.1-8b-instant', k: int = 3):
    
    retriever = vs.as_retriever(search_kwargs={'k': k})
    
    llm = ChatGroq(
        model=model_name,
        temperature=0,
        api_key=GROQ_API_KEY
    )
    
    prompt = ChatPromptTemplate.from_template(
    """
    Kamu adalah asisten yang membantu menjawab pertanyaan berdasarkan informasi yang diberikan.
    Gunakan HANYA informasi dari context di bawah untuk menjawab.
    Jika informasi tidak tersedia dalam context, katakan "Saya tidak menemukan informasi tersebut dalam knowledge base."

    Context:
    {context}

    Pertanyaan: {question}

    Jawaban:
    """
    )
    
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {
            "context": retriever | format_docs,  
            "question": RunnablePassthrough()    
        }
        | prompt           
        | llm              
        | StrOutputParser()
    )    
    
    print('RAG berhasil di buat')
    return rag_chain

rag_chain = build_rag(vector_store)

RAG berhasil di buat


## interaksi dengan RAG

### update knowledge


In [13]:
add_to_knowledge_base("""
                    Berdasarkan dokumen yang digunakan, faktor sosial memiliki peran penting dalam memengaruhi tingkat kemiskinan di Indonesia. Salah satu faktor utama adalah kualitas pembangunan manusia yang tercermin dalam Indeks Pembangunan Manusia (IPM), yang meliputi aspek pendidikan, kesehatan, dan standar hidup. Rendahnya kualitas pendidikan dan kesehatan menyebabkan produktivitas masyarakat menjadi rendah, sehingga berdampak pada rendahnya pendapatan dan meningkatnya risiko kemiskinan. Sebaliknya, peningkatan kualitas pembangunan manusia terbukti mampu menurunkan tingkat kemiskinan secara signifikan.
                    Selain itu, kemiskinan juga dipengaruhi oleh fenomena lingkaran setan kemiskinan (vicious circle of poverty), di mana rendahnya kualitas sumber daya manusia menyebabkan produktivitas yang rendah, yang kemudian berdampak pada rendahnya pendapatan, tabungan, dan investasi. Kondisi ini terus berulang dan membuat masyarakat sulit keluar dari kemiskinan. Di sisi lain, ketimpangan dalam pembangunan juga turut memperparah kondisi tersebut, karena tidak semua kelompok masyarakat memiliki akses yang sama terhadap pendidikan, layanan kesehatan, dan peluang ekonomi.
                    Lebih lanjut, keterbatasan akses terhadap sumber daya, termasuk akibat kebijakan tertentu, juga menjadi faktor sosial yang berpengaruh terhadap kemiskinan. Ketika masyarakat tidak memiliki akses yang memadai terhadap sumber daya yang dapat menunjang kehidupan mereka, maka kemampuan untuk memenuhi kebutuhan dasar menjadi terhambat. Dengan demikian, faktor sosial seperti kualitas pembangunan manusia, ketimpangan akses, serta keterbatasan sumber daya memiliki keterkaitan erat dalam membentuk dan mempertahankan kondisi kemiskinan di Indonesia.  
                    """)

vector_store = build_vector_store(knowledge_base)

rag_chain = build_rag(vector_store)

print('knowledge base dan RAG chain berhasil di perbarui')

knowledge_base updated. total dokumen : 7
jumlah chunk yang dihasilkan: 18
vector berhasil di bangun
RAG berhasil di buat
knowledge base dan RAG chain berhasil di perbarui


In [16]:
import textwrap

def ask(question: str, chain):
    print("\n" + "=" * 60)
    print(f"pertanyaan: {question}")
    print("=" * 60)

    answer = chain.invoke(question)

    print("jawaban:")
    print("\n".join(textwrap.wrap(answer, width=100)))
    print("=" * 60 + "\n")

    return answer

def chat_loop(chain):
    
    while True:
        question = input(" masukan pertanyaan (ketik 'stop' untuk keluar):")
        if question.lower() == 'stop':
            print('terimakasih telah menggunakan layanan ini')
            break
        ask(question, chain)
        
chat_loop(rag_chain)


pertanyaan: sebutkan 3 faktor penyebab kemiskinan di indonesia
jawaban:
Berdasarkan dokumen yang digunakan, saya dapat menyebutkan 3 faktor penyebab kemiskinan di Indonesia
sebagai berikut:  1. Rendahnya kualitas pendidikan 2. Rendahnya kualitas kesehatan 3. Produktivitas
masyarakat yang rendah


pertanyaan: hari raya indonesia
jawaban:
Saya tidak menemukan informasi tentang "hari raya Indonesia" dalam konteks yang diberikan.


pertanyaan: sebutkan 3 faktor kemiskinan di indonesia dalam sosial
jawaban:
Berdasarkan dokumen yang digunakan, ada tiga faktor kemiskinan di Indonesia dalam aspek sosial,
yaitu:  1. Kualitas pembangunan manusia yang tercermin dalam Indeks Pembangunan Manusia (IPM), yang
meliputi aspek pendidikan, kesehatan, dan standar hidup. 2. Keterbatasan akses terhadap sumber daya,
termasuk akibat kebijakan tertentu. 3. Ketimpangan akses terhadap sumber daya yang dapat menunjang
kehidupan masyarakat.

terimakasih telah menggunakan layanan ini


# 5. Evaluasi & Analisis Sistem
1. Lakukan evaluasi terhadap aplikasi RAG yang telah dibuat dengan menguji beberapa pertanyaan.
2. Buat text cell terpisah, lalu jelaskan hasil evaluasi termasuk keterbatasan atau kekurangan dari sistem yang dikembangkan.


| No | Pertanyaan | Jawaban RAG | Terjawab |
|----|-----------|-------------|-----------|
| 1  | sebutkan 3 faktor penyebab kemiskinan di indonesia       | Berdasarkan dokumen yang digunakan, saya dapat menyebutkan 3 faktor penyebab kemiskinan di Indonesia sebagai berikut:  1. Rendahnya kualitas pendidikan 2. Rendahnya kualitas kesehatan 3. Produktivitas masyarakat yang rendah       | ✅    |
| 2  |   hari raya indonesia      |  Saya tidak menemukan informasi tentang "hari raya Indonesia" dalam konteks yang diberikan.        |  ❌   |
| 3  | sebutkan 3 faktor kemiskinan di indonesia dalam sosial       | Berdasarkan dokumen yang digunakan, ada tiga faktor kemiskinan di Indonesia dalam aspek sosial,yaitu:  1. Kualitas pembangunan manusia yang tercermin dalam Indeks Pembangunan Manusia (IPM), yang kehidupan masyarakat.         | ✅ |


analisis hasil jawaban:
- sistem berhasil menjawab pertanyaan yang terdapat di knowlage base
- sistem tidak dapat menjawab dan otomatis menjawab "Saya tidak menemukan informasi tentang "pertanyaan" dalam konteks yang diberikan" ketika terdapat pertanyaan yang tidak terdapat di knowledge.

# 6. Insight & Recommendation Action
1. Tuliskan insight yang diperoleh dari hasil implementasi dan evaluasi sistem.
2. Berikan rekomendasi pengembangan lanjutan yang dapat dilakukan untuk meningkatkan kualitas aplikasi RAG.


### Insight:
- RAG terbukti efektif untuk membatasi LLM agar hanya menjawab berdasarkan data spesifik,
  sehingga mengurangi **hallucination** dibandingkan LLM tanpa context.
- Kualitas jawaban sangat bergantung pada kualitas dan cakupan knowledge base.
- Pemilihan chunk_size berpengaruh besar: terlalu kecil = konteks hilang, terlalu besar = noise tinggi.


### Rekomendasi Pengembangan:
1. **Hybrid Search** — gabungkan similarity search (dense) dengan keyword search (sparse/BM25) untuk retrieval lebih akurat.
2. **Reranker** — tambahkan reranking model (misal: Cohere Rerank) untuk mengurutkan ulang dokumen hasil retrieval.
3. **Evaluasi Kuantitatif** — gunakan framework seperti RAGAS untuk mengukur faithfulness, answer relevancy, dan context precision secara otomatis.
4. **Persistent Vector Store** — simpan FAISS index ke disk (`vector_store.save_local()`) agar tidak perlu rebuild setiap sesi.

# Reflection Question
1. Setelah menyelesaikan assignment ini, jawablah pertanyaan berikut.
Setelah membangun aplikasi RAG, bagian mana dari proses pengambilan informasi (retrieval) yang membuatmu paling memahami pentingnya kualitas knowledge base dalam menghasilkan jawaban yang relevan? Jelaskan bagaimana pengalaman ini mengubah caramu melihat proses pencarian informasi dalam sebuah chatbot.
2. Dalam memilih LLM, embedding model, dan vector database, apa hal yang membuatmu menyadari bahwa arsitektur teknis sebuah sistem RAG harus disesuaikan dengan kebutuhan pengguna dan konteks data? Jelaskan bagaimana keputusan teknismu dipengaruhi oleh karakteristik topik knowledge base yang kamu gunakan.


1. Setelah membangun aplikasi RAG, bagian yang paling membuat saya memahami pentingnya kualitas knowledge base adalah pada proses retrieval, khususnya saat sistem mengambil chunk yang relevan dari vector database. Pada tahap ini terlihat jelas bahwa kualitas jawaban sangat bergantung pada bagaimana isi knowledge base disusun dan dipotong menjadi bagian-bagian kecil. Ketika knowledge base berisi informasi yang kurang jelas, terlalu umum, atau tidak terstruktur dengan baik, maka retriever sering mengambil konteks yang kurang relevan, sehingga jawaban yang dihasilkan oleh model menjadi tidak tepat atau bahkan melenceng. Sebaliknya, ketika knowledge base disusun dengan ringkasan yang jelas, terfokus, dan dipisahkan berdasarkan topik tertentu, hasil retrieval menjadi jauh lebih akurat. Hal ini menunjukkan bahwa model LLM sebenarnya hanya “sebaik” konteks yang diberikan kepadanya. Dari pengalaman ini, cara pandang saya terhadap chatbot berubah, di mana sebelumnya saya menganggap bahwa kecerdasan utama terletak pada model LLM, namun ternyata kualitas data dan struktur informasi memiliki peran yang sama penting, bahkan bisa menjadi penentu utama dalam menghasilkan jawaban yang relevan.
2. Dalam proses memilih LLM, model embedding, dan vector database, saya menyadari bahwa arsitektur teknis pada sistem RAG tidak bisa dibuat secara umum, melainkan harus disesuaikan dengan kebutuhan pengguna dan karakteristik data yang digunakan. Hal ini terlihat dari bagaimana setiap komponen memiliki peran yang berbeda dan saling memengaruhi hasil akhir.